In [23]:
import numpy as np
import pandas as pd

## Workflow
### 1. Encoding
encode_atc()
encode_drug()

### 2. ATC Code-pair similarity
atc_code_similarity()
show_atc_code_similarity()

### 3. Drug-drug pair similarity
drug_similarity()



In [31]:
# --- Demo ---
drugs = {
    # --- Single ATC code ---
    "metformin":           ["A10BA02"],          # biguanide
    "glipizide":           ["A10BB07"],          # sulfonylurea
    "glibenclamide":       ["A10BB01"],          # sulfonylurea
    "insulin_glargine":    ["A10AE04"],          # long-acting insulin
    "enalapril":           ["C09AA02"],          # ACE inhibitor

    # --- Multiple ATC codes ---
    "metformin_glibenc":   ["A10BA02", "A10BB01"],  # fixed-dose combo: shares codes with both metformin and glibenclamide
    "nateglinide":         ["A10BX03", "A10BB12"],  # dual-coded antidiabetic
    "enalapril_hctz":      ["C09BA02", "C03AA03"],  # ACE inhibitor + diuretic combo
    "hydrochlorothiazide": ["C03AA03", "C07BB03"],  # diuretic; shares C03AA03 with enalapril_hctz
}

In [36]:
# Encode each ATC code
def encode_drug(atc_codes: list[str]) -> list[list[str]]:
    """
    Encode a drug's ATC code(s) into hierarchical level lists (level 2-4).
    """
    cuts = [3, 4, 5]
    return [[code[:c] for c in cuts if len(code) >= c] for code in atc_codes]

In [37]:
drugs_encoded = {
    name: encode_drug(atc_codes)
    for name, atc_codes in drugs.items()
}

In [38]:
for name, atc_encodings in drugs_encoded.items():
    print(f"{name}")
    for atc_encoding in atc_encodings: 
        print(f"{atc_encoding}")

metformin
['A10', 'A10B', 'A10BA']
glipizide
['A10', 'A10B', 'A10BB']
glibenclamide
['A10', 'A10B', 'A10BB']
insulin_glargine
['A10', 'A10A', 'A10AE']
enalapril
['C09', 'C09A', 'C09AA']
metformin_glibenc
['A10', 'A10B', 'A10BA']
['A10', 'A10B', 'A10BB']
nateglinide
['A10', 'A10B', 'A10BX']
['A10', 'A10B', 'A10BB']
enalapril_hctz
['C09', 'C09B', 'C09BA']
['C03', 'C03A', 'C03AA']
hydrochlorothiazide
['C03', 'C03A', 'C03AA']
['C07', 'C07B', 'C07BB']


In [39]:
def atc_code_similarity(levels_a: list[str], levels_b: list[str]) -> float:
    """
    Compute hierarchical path similarity between two encoded ATC level lists.
    Wegith at depth d (1-indexed): 2^(d-1)
    Doniminator: sum of all weights = 2^0 + 2^1 + 2^2 = 7

    Return a score in [0,1].
    """
    assert len(levels_a) == len(levels_b) == 3

    weights = [2**d for d in range(3)] # [1,2,4]
    total = sum(weights)               # 7

    score = sum(
        w*(a==b)
        for w, a, b in zip(weights, levels_a, levels_b)
    )
    return score/total

In [40]:
### Only for display atc code similarity results, not contributing to the actuall workflow
def show_atc_score_matrix(P, Q, drugs, drugs_encoded):
    """
    Display code-pair similarity scores as a matrix. 
    Rows = ATC codes of drug P, Columns = ATC codes of drug Q.
    """

    codes_P = drugs[P]
    codes_Q = drugs[Q]
    encoded_P = drugs_encoded[P]
    encoded_Q = drugs_encoded[Q]

    # Using nested list comprehension
    matrix = [
        [atc_code_similarity(atc_path_p, atc_path_q) for atc_path_q in encoded_Q]
        for atc_path_p in encoded_P
    ]

    df = pd.DataFrame(matrix, index = codes_P, columns = codes_Q).round(4)
    df.index.name = P
    df.columns.name = Q

    print(f"n{P} vs {Q}")
    print(df.to_string())
    print()
    return df

In [43]:
def drug_similarity(encoded_P: list[list[str]],
                    encoded_Q: list[list[str]],
                    verbose: bool = False) -> float:
    """
    Compute symmetrized mean-of-max drug-drug ATC similarity.

    Parameters
    ----------
    encoded_P : list of level lists for drug P
    encoded_Q : list of level lists for drug Q
    verbose   : if True, prints the score matrix and intermediate values

    Returns
    -------
    float in [0, 1]
    """
    m, n = len(encoded_P), len(encoded_Q)

    # Build m x n score matrix
    S = np.array([
        [atc_code_similarity(p, q) for q in encoded_Q]
        for p in encoded_P
    ])

    sim_P_to_Q = S.max(axis=1).mean()   # row-wise max, then average
    sim_Q_to_P = S.max(axis=0).mean()   # col-wise max, then average
    sym_sim    = (sim_P_to_Q + sim_Q_to_P) / 2

    if verbose:
        print(f"Score matrix S ({m} x {n}):")
        header = "       " + "  ".join(f"q{j+1}" for j in range(n))
        print(header)
        for i, row in enumerate(S):
            print(f"  p{i+1}   " + "  ".join(f"{v:.4f}" for v in row))
        print()
        print(f"  Row maxima (P→Q): {S.max(axis=1).round(4).tolist()}")
        print(f"  Sim(P→Q)        = {sim_P_to_Q:.4f}")
        print()
        print(f"  Col maxima (Q→P): {S.max(axis=0).round(4).tolist()}")
        print(f"  Sim(Q→P)        = {sim_Q_to_P:.4f}")
        print()
        print(f"  Sim(P,Q) = ({sim_P_to_Q:.4f} + {sim_Q_to_P:.4f}) / 2 = {sym_sim:.4f}")

    return sym_sim

In [41]:
# --- examples ---
pairs = [
    # --- single vs single ---
    ("glipizide","glibenclamide"),
    ("metformin","glipizide"),
    ("metformin","insulin_glargine"),
    ("metformin","enalapril"),
    # --- single vs multi ---
    ("metformin",          "metformin_glibenc"),
    ("metformin",          "nateglinide"),      
    ("insulin_glargine",   "metformin_glibenc"),
    ("enalapril",          "enalapril_hctz"), 
    # --- multi vs multi ---
    ("metformin_glibenc",  "nateglinide"),      
    ("enalapril_hctz",     "hydrochlorothiazide"), 
]

In [47]:
for P, Q in pairs:
    show_atc_score_matrix(P, Q, drugs, drugs_encoded)
    drug_similarity(drugs_encoded[P],drugs_encoded[Q], verbose=True)
    print("-"*10)

nglipizide vs glibenclamide
glibenclamide  A10BB01
glipizide             
A10BB07            1.0

Score matrix S (1 x 1):
       q1
  p1   1.0000

  Row maxima (P→Q): [1.0]
  Sim(P→Q)        = 1.0000

  Col maxima (Q→P): [1.0]
  Sim(Q→P)        = 1.0000

  Sim(P,Q) = (1.0000 + 1.0000) / 2 = 1.0000
----------
nmetformin vs glipizide
glipizide  A10BB07
metformin         
A10BA02     0.4286

Score matrix S (1 x 1):
       q1
  p1   0.4286

  Row maxima (P→Q): [0.4286]
  Sim(P→Q)        = 0.4286

  Col maxima (Q→P): [0.4286]
  Sim(Q→P)        = 0.4286

  Sim(P,Q) = (0.4286 + 0.4286) / 2 = 0.4286
----------
nmetformin vs insulin_glargine
insulin_glargine  A10AE04
metformin                
A10BA02            0.1429

Score matrix S (1 x 1):
       q1
  p1   0.1429

  Row maxima (P→Q): [0.1429]
  Sim(P→Q)        = 0.1429

  Col maxima (Q→P): [0.1429]
  Sim(Q→P)        = 0.1429

  Sim(P,Q) = (0.1429 + 0.1429) / 2 = 0.1429
----------
nmetformin vs enalapril
enalapril  C09AA02
metformin         
